
# Chapter 12: Modern Methodology

Source span: printed pages 267-282; PDF pages 282-297. The source was inspected for section structure and terminology only. The prose, examples, computations, diagrams, and artifacts below are original course material.

## Chapter Goal

The chapter collects methods that became important once directional data analysis moved beyond small closed-form tests: outlier diagnostics, robust estimation, bootstrap confidence regions, nonparametric density estimation, Bayesian shrinkage, and smoothing of spherical responses. The common theme is that every modern method must still respect the sample space. A bootstrap sample must be resampled on the sphere, not in a tangent plane by default. A robust estimator must downweight angular contamination without losing unit length. A density estimate must integrate over the sphere. A smoothing curve must return directions, not arbitrary three-dimensional vectors.

This notebook turns those ideas into a compact lab. We start with a synthetic spherical sample so the geometry is visible and reproducible. Then we add controlled contamination, compare ordinary and robust mean directions, build bootstrap confidence cones, estimate a spherical density with von Mises-Fisher kernels, show Bayesian shrinkage as pseudo-resultant geometry, and smooth a noisy spherical trajectory. Each visual has a nearby invariant check because modern methodology is easy to misuse when it is treated as ordinary Euclidean data analysis with a normalization step at the end.



## Visualization Storyboard And Library Routing

- **Outlier influence atlas.** A single contaminating direction is moved around a great circle while the mean resultant and mean direction are recomputed. Matplotlib is best here because the audit target is a durable two-panel influence curve plus a labeled sphere scatter.

- **Robust direction and bootstrap cone.** A Huber-style angular M-estimator is compared with the ordinary mean. Bootstrap resamples of the mean direction are drawn as angular deviations from the estimate, producing an empirical confidence cone. This visual teaches why bootstrap methods are especially useful when concentration is low or sample size is modest.

- **Kernel density on the sphere.** A von Mises-Fisher kernel estimate is evaluated on a sphere grid. The figure shows both a longitude-latitude heatmap and a colored sphere. The check estimates the integral over the sphere by deterministic quadrature.

- **Bayesian shrinkage diagram.** A prior direction is represented as a pseudo-resultant vector. Adding it to the sample resultant produces a posterior-style direction. This is not a full Bayesian analysis; it is the geometric core of conjugate-looking shrinkage.

- **Spherical smoothing curve.** A scalar predictor with spherical response is smoothed by extrinsic local averaging followed by renormalization. Plotly is useful because rotating the curve exposes whether the fitted path remains on the sphere.

The chapter uses NumPy/SciPy for simulation and vector algebra, Matplotlib for static artifacts, Plotly for the rotatable smoothing lab, and course-local artifact helpers for reproducible outputs.


In [ ]:

from pathlib import Path
import sys


def find_book_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (
            (candidate / "AGENTS.md").exists()
            and (candidate / "scripts" / "validate_dirstats_course.py").exists()
            and (candidate / "utils").exists()
        ):
            return candidate
    raise RuntimeError("Could not locate Directional-Statistics course root")


BOOK_ROOT = find_book_root(Path.cwd())
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

TOPIC = "chapter-12"
source_span = {"printed": "267-282", "pdf": "282-297"}
print(f"Course root: {BOOK_ROOT}")
source_span


In [ ]:

import json
import math

import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
import numpy as np
import plotly.graph_objects as go
from scipy import stats

from utils.artifacts import display_artifact, save_json, save_matplotlib, save_plotly_html
from utils.validation import assert_artifacts

rng = np.random.default_rng(1212)
np.set_printoptions(precision=5, suppress=True)
plt.rcParams.update({
    "figure.dpi": 130,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
})


def normalize(v, axis=-1, eps=1e-12):
    v = np.asarray(v, dtype=float)
    n = np.linalg.norm(v, axis=axis, keepdims=True)
    return v / np.maximum(n, eps)


def sample_uniform_sphere(count):
    z = rng.uniform(-1.0, 1.0, size=count)
    phi = rng.uniform(0.0, 2.0 * np.pi, size=count)
    r = np.sqrt(np.maximum(0.0, 1.0 - z * z))
    return np.column_stack([r * np.cos(phi), r * np.sin(phi), z])


def sample_vmf(mu, kappa, count):
    mu = normalize(mu)
    if kappa <= 1e-12:
        base = sample_uniform_sphere(count)
    else:
        u = rng.uniform(size=count)
        z = (1.0 / kappa) * np.log(np.exp(-kappa) + u * (np.exp(kappa) - np.exp(-kappa)))
        phi = rng.uniform(0.0, 2.0 * np.pi, size=count)
        r = np.sqrt(np.maximum(0.0, 1.0 - z * z))
        base = np.column_stack([r * np.cos(phi), r * np.sin(phi), z])
    north = np.array([0.0, 0.0, 1.0])
    if np.linalg.norm(mu - north) < 1e-10:
        return base
    axis = np.cross(north, mu)
    axis = axis / np.linalg.norm(axis)
    angle = math.acos(float(np.clip(np.dot(north, mu), -1.0, 1.0)))
    K = np.array([[0, -axis[2], axis[1]], [axis[2], 0, -axis[0]], [-axis[1], axis[0], 0]])
    R = np.eye(3) + math.sin(angle) * K + (1.0 - math.cos(angle)) * (K @ K)
    return base @ R.T


def mean_direction(points):
    resultant = points.sum(axis=0)
    return normalize(resultant), float(np.linalg.norm(resultant) / len(points))


def angular_distance(a, b):
    return np.arccos(np.clip(np.sum(normalize(a) * normalize(b), axis=-1), -1.0, 1.0))


def robust_direction(points, cutoff=0.55, iterations=20):
    direction, _ = mean_direction(points)
    for _ in range(iterations):
        angles = angular_distance(points, direction)
        weights = np.minimum(1.0, cutoff / np.maximum(angles, 1e-6))
        direction = normalize((weights[:, None] * points).sum(axis=0))
    return direction, weights


def fibonacci_sphere(n=6000):
    i = np.arange(n, dtype=float)
    z = 1.0 - 2.0 * (i + 0.5) / n
    theta = np.pi * (3.0 - np.sqrt(5.0)) * i
    r = np.sqrt(np.maximum(0.0, 1.0 - z * z))
    return np.column_stack([r * np.cos(theta), r * np.sin(theta), z])


def vmf_c3(kappa):
    if kappa <= 1e-12:
        return 1.0 / (4.0 * np.pi)
    return kappa / (4.0 * np.pi * np.sinh(kappa))


def spherical_kde(points, grid, kappa=12.0):
    dots = grid @ points.T
    return vmf_c3(kappa) * np.exp(kappa * dots).mean(axis=1)


def sphere_wire(ax, alpha=0.18):
    u = np.linspace(0, 2*np.pi, 40)
    v = np.linspace(0, np.pi, 20)
    U, V = np.meshgrid(u, v)
    X = np.cos(U)*np.sin(V); Y = np.sin(U)*np.sin(V); Z = np.cos(V)
    ax.plot_wireframe(X, Y, Z, color="#9aa0a6", linewidth=0.35, alpha=alpha)
    ax.set_box_aspect((1, 1, 1))
    ax.set_axis_off()



## Outliers Are Angular Influence Problems

On a line, a single outlier can be far away in magnitude. On a sphere, every observation has unit length, so contamination is angular rather than radial. A point near the antipode of the main cluster can still pull the resultant direction and reduce the mean resultant length. The first artifact moves one contaminating point around a great circle and records what happens to the ordinary mean.

The inspection target is the split between direction and concentration. The mean direction may move only moderately, while the mean resultant length can drop sharply when an observation lands opposite the cluster. This is why outlier diagnostics in the chapter focus on how much each observation changes the resultant or the fitted direction, not on Euclidean distance from the origin.


In [ ]:

true_mu = normalize(np.array([0.35, -0.15, 0.925]))
clean = sample_vmf(true_mu, 9.0, 34)
clean_mean, clean_rbar = mean_direction(clean)

sweep_angles = np.linspace(0.0, 2.0 * np.pi, 181)
# Great circle through true_mu and a perpendicular direction.
basis_a = true_mu
basis_b = normalize(np.array([basis_a[1], -basis_a[0], 0.0]))
contaminants = np.cos(sweep_angles)[:, None] * basis_a + np.sin(sweep_angles)[:, None] * basis_b
mean_shifts = []
rbar_values = []
for outlier in contaminants:
    contaminated = np.vstack([clean, outlier])
    m, r = mean_direction(contaminated)
    mean_shifts.append(np.degrees(angular_distance(m, clean_mean)))
    rbar_values.append(r)
mean_shifts = np.array(mean_shifts)
rbar_values = np.array(rbar_values)
worst_index = int(np.argmax(mean_shifts))
contaminated = np.vstack([clean, contaminants[worst_index]])
cont_mean, cont_rbar = mean_direction(contaminated)

fig = plt.figure(figsize=(13, 5.3))
grid = fig.add_gridspec(1, 3, width_ratios=[1.05, 1, 1])
ax0 = fig.add_subplot(grid[0, 0], projection="3d")
sphere_wire(ax0, alpha=0.22)
ax0.scatter(clean[:, 0], clean[:, 1], clean[:, 2], color="#345995", s=24, label="clean sample")
ax0.scatter(*contaminants[worst_index], color="#E76F51", s=80, label="largest shift contaminant")
ax0.quiver(0, 0, 0, *clean_mean, color="#2A9D8F", linewidth=2.5, label="clean mean")
ax0.quiver(0, 0, 0, *cont_mean, color="#B84A62", linewidth=2.5, label="contaminated mean")
ax0.set_title("One angular contaminant on S^2")
ax0.legend(loc="lower left", fontsize=7)

ax1 = fig.add_subplot(grid[0, 1])
ax1.plot(np.degrees(sweep_angles), mean_shifts, color="#B84A62")
ax1.axvline(np.degrees(sweep_angles[worst_index]), color="#555555", linestyle="--", linewidth=1)
ax1.set_title("Mean-direction shift")
ax1.set_xlabel("contaminant great-circle angle")
ax1.set_ylabel("shift from clean mean (degrees)")

ax2 = fig.add_subplot(grid[0, 2])
ax2.plot(np.degrees(sweep_angles), rbar_values, color="#345995")
ax2.axhline(clean_rbar, color="#2A9D8F", linestyle="--", linewidth=1, label="clean Rbar")
ax2.set_title("Mean resultant length")
ax2.set_xlabel("contaminant great-circle angle")
ax2.set_ylabel("Rbar after adding one point")
ax2.legend()
fig.suptitle("Outlier diagnostics on a sphere: direction shift and resultant loss", y=1.03, fontsize=14)
fig.tight_layout()
outlier_path = save_matplotlib(fig, TOPIC, "outliers", "angular-outlier-influence-atlas.png")
plt.close(fig)
display_artifact(outlier_path, width=940)

outlier_checks = {
    "clean_unit_norm_max_error": float(np.max(np.abs(np.linalg.norm(clean, axis=1) - 1.0))),
    "largest_mean_shift_degrees": float(mean_shifts.max()),
    "rbar_decreases_for_antipodal_contamination": bool(rbar_values.min() < clean_rbar),
    "contaminated_mean_unit_norm_error": float(abs(np.linalg.norm(cont_mean) - 1.0)),
}
outlier_checks



## Robust Direction And Bootstrap Confidence Cone

A robust mean direction should move less under the same contamination. The simple M-estimator below repeatedly downweights observations whose angular distance from the current direction is larger than a cutoff. This is not the only robust estimator in the literature, but it exposes the chapter's practical point: robustness should be stated as an angular influence rule.

The bootstrap portion resamples the contaminated data on the sphere and recomputes the ordinary mean direction. The empirical distribution of angular deviations gives a confidence cone radius. This keeps the uncertainty statement on the sphere. The histogram is not an ordinary standard-error bar; it is a distribution of angles away from an estimated direction.


In [ ]:

robust_mean, robust_weights = robust_direction(contaminated, cutoff=0.45)
ordinary_shift = np.degrees(angular_distance(cont_mean, clean_mean))
robust_shift = np.degrees(angular_distance(robust_mean, clean_mean))

boot_count = 900
boot_dirs = []
for _ in range(boot_count):
    sample = contaminated[rng.integers(0, len(contaminated), size=len(contaminated))]
    boot_dirs.append(mean_direction(sample)[0])
boot_dirs = np.array(boot_dirs)
boot_angles = np.degrees(angular_distance(boot_dirs, cont_mean))
cone_90 = float(np.quantile(boot_angles, 0.90))
cone_95 = float(np.quantile(boot_angles, 0.95))

fig = plt.figure(figsize=(13, 5.2))
grid = fig.add_gridspec(1, 3, width_ratios=[1.05, 1, 1])
ax0 = fig.add_subplot(grid[0, 0], projection="3d")
sphere_wire(ax0, alpha=0.2)
ax0.scatter(contaminated[:, 0], contaminated[:, 1], contaminated[:, 2], c=robust_weights, cmap="viridis", s=34)
ax0.quiver(0, 0, 0, *clean_mean, color="#2A9D8F", linewidth=2.2)
ax0.quiver(0, 0, 0, *cont_mean, color="#B84A62", linewidth=2.2)
ax0.quiver(0, 0, 0, *robust_mean, color="#F4A261", linewidth=2.2)
ax0.set_title("Weights for robust angular mean")

ax1 = fig.add_subplot(grid[0, 1])
ax1.bar(["ordinary", "robust"], [ordinary_shift, robust_shift], color=["#B84A62", "#F4A261"])
ax1.set_title("Shift from clean mean")
ax1.set_ylabel("degrees")

ax2 = fig.add_subplot(grid[0, 2])
ax2.hist(boot_angles, bins=32, color="#345995", alpha=0.82)
ax2.axvline(cone_90, color="#F4A261", linewidth=2, label=f"90% cone {cone_90:.1f} deg")
ax2.axvline(cone_95, color="#B84A62", linewidth=2, label=f"95% cone {cone_95:.1f} deg")
ax2.set_title("Bootstrap angular deviations")
ax2.set_xlabel("angle from sample mean (degrees)")
ax2.legend()
fig.suptitle("Robust direction and bootstrap confidence cone", y=1.03, fontsize=14)
fig.tight_layout()
bootstrap_path = save_matplotlib(fig, TOPIC, "bootstrap-robust", "robust-direction-bootstrap-cone.png")
plt.close(fig)
display_artifact(bootstrap_path, width=940)

bootstrap_checks = {
    "robust_moves_less_than_ordinary": bool(robust_shift < ordinary_shift),
    "robust_mean_unit_norm_error": float(abs(np.linalg.norm(robust_mean) - 1.0)),
    "bootstrap_direction_unit_norm_max_error": float(np.max(np.abs(np.linalg.norm(boot_dirs, axis=1) - 1.0))),
    "cone_95_degrees": cone_95,
    "cone_quantiles_ordered": bool(0.0 < cone_90 < cone_95 < 90.0),
}
bootstrap_checks



## Kernel Density Estimation On The Sphere

A spherical kernel density estimate replaces each observation by a small von Mises-Fisher bump and averages the bumps. The smoothing parameter is a concentration: larger values make tighter bumps; smaller values spread mass more broadly. The important constraint is that the estimate is a density on the sphere, so it should integrate to one over surface area.

The figure uses a deterministic grid. The left panel is an equirectangular heatmap for reading coordinates. The right panel puts the same density on the sphere so the learner sees that high density is a cap-like region, not a rectangle. The numerical check uses a Fibonacci sphere quadrature to estimate the integral.


In [ ]:

# Use the clean cluster plus a smaller secondary cluster to make a density ridge visible.
secondary = sample_vmf(normalize(np.array([-0.55, 0.62, 0.56])), 7.0, 18)
density_sample = np.vstack([clean, secondary])
lon = np.linspace(-np.pi, np.pi, 160)
lat = np.linspace(-np.pi / 2, np.pi / 2, 80)
Lon, Lat = np.meshgrid(lon, lat)
Grid = np.column_stack([np.cos(Lat).ravel() * np.cos(Lon).ravel(), np.cos(Lat).ravel() * np.sin(Lon).ravel(), np.sin(Lat).ravel()])
density_values = spherical_kde(density_sample, Grid, kappa=14.0).reshape(Lat.shape)
quad_grid = fibonacci_sphere(9000)
integral_estimate = float(4.0 * np.pi * spherical_kde(density_sample, quad_grid, kappa=14.0).mean())

fig = plt.figure(figsize=(13, 5.4))
ax0 = fig.add_subplot(1, 2, 1)
mesh = ax0.pcolormesh(np.degrees(lon), np.degrees(lat), density_values, shading="auto", cmap="magma")
ax0.scatter(np.degrees(np.arctan2(density_sample[:, 1], density_sample[:, 0])), np.degrees(np.arcsin(density_sample[:, 2])), s=10, color="white", alpha=0.65)
ax0.set_title("Longitude-latitude view of spherical KDE")
ax0.set_xlabel("longitude")
ax0.set_ylabel("latitude")
fig.colorbar(mesh, ax=ax0, shrink=0.8, label="density")

ax1 = fig.add_subplot(1, 2, 2, projection="3d")
colors = plt.cm.magma((density_values - density_values.min()) / (density_values.max() - density_values.min()))
X = np.cos(Lat) * np.cos(Lon); Y = np.cos(Lat) * np.sin(Lon); Z = np.sin(Lat)
ax1.plot_surface(X, Y, Z, facecolors=colors, linewidth=0, antialiased=False, shade=False, alpha=0.95)
ax1.scatter(density_sample[:, 0], density_sample[:, 1], density_sample[:, 2], color="white", edgecolor="#23343b", s=18)
ax1.set_title("Same density on S^2")
ax1.set_axis_off(); ax1.set_box_aspect((1, 1, 1))
fig.suptitle("Spherical kernel density estimate: smooth mass without flattening the sphere", y=1.02, fontsize=14)
fig.tight_layout()
density_path = save_matplotlib(fig, TOPIC, "density", "spherical-kernel-density-estimate.png")
plt.close(fig)
display_artifact(density_path, width=940)

density_checks = {
    "density_positive": bool(np.all(density_values > 0)),
    "density_integral_estimate": integral_estimate,
    "density_integral_abs_error": float(abs(integral_estimate - 1.0)),
    "density_sample_unit_norm_max_error": float(np.max(np.abs(np.linalg.norm(density_sample, axis=1) - 1.0))),
}
density_checks



## Bayesian Shrinkage As Pseudo-Resultant Geometry

The chapter only gives a brief doorway into Bayesian directional methods, but a useful geometric picture is available without a full posterior sampler. A prior mean direction can be represented by a pseudo-resultant vector with strength `tau`. The sample contributes its resultant vector. The posterior-style direction points along the vector sum.

This diagram teaches two habits. First, the prior should be a direction and a strength, not an unconstrained coordinate vector. Second, the influence of the prior should be visible as a vector addition on the sphere. With a large sample or a concentrated sample, the likelihood resultant dominates; with a weak or diffuse sample, the prior can visibly shrink the estimate.


In [ ]:

prior_mu = normalize(np.array([0.0, 0.9, 0.44]))
tau_values = np.array([0.0, 2.0, 6.0, 14.0, 30.0])
sample_resultant = contaminated.sum(axis=0)
posterior_dirs = np.array([normalize(sample_resultant + tau * prior_mu) for tau in tau_values])
prior_angles = np.degrees(angular_distance(posterior_dirs, prior_mu))
sample_angles = np.degrees(angular_distance(posterior_dirs, cont_mean))

fig = plt.figure(figsize=(12.5, 5.2))
ax0 = fig.add_subplot(1, 2, 1, projection="3d")
sphere_wire(ax0, alpha=0.18)
ax0.scatter(contaminated[:, 0], contaminated[:, 1], contaminated[:, 2], color="#345995", s=18, alpha=0.75)
ax0.quiver(0, 0, 0, *cont_mean, color="#B84A62", linewidth=2.5, label="sample mean")
ax0.quiver(0, 0, 0, *prior_mu, color="#2A9D8F", linewidth=2.5, label="prior direction")
for tau, direction in zip(tau_values[1:], posterior_dirs[1:]):
    ax0.quiver(0, 0, 0, *direction, color="#F4A261", linewidth=1.4, alpha=min(1.0, 0.35 + tau/35))
ax0.set_title("Prior pseudo-resultant plus sample resultant")
ax0.legend(loc="lower left", fontsize=8)

ax1 = fig.add_subplot(1, 2, 2)
ax1.plot(tau_values, sample_angles, marker="o", label="angle from sample mean", color="#B84A62")
ax1.plot(tau_values, prior_angles, marker="o", label="angle from prior direction", color="#2A9D8F")
ax1.set_title("Shrinkage path as prior strength grows")
ax1.set_xlabel("prior pseudo-count strength tau")
ax1.set_ylabel("angular distance (degrees)")
ax1.legend()
fig.suptitle("Bayesian directional shrinkage as vector-resultant geometry", y=1.03, fontsize=14)
fig.tight_layout()
shrinkage_path = save_matplotlib(fig, TOPIC, "bayesian-shrinkage", "prior-pseudo-resultant-shrinkage.png")
plt.close(fig)
display_artifact(shrinkage_path, width=920)

shrinkage_checks = {
    "posterior_dirs_unit_norm_max_error": float(np.max(np.abs(np.linalg.norm(posterior_dirs, axis=1) - 1.0))),
    "posterior_moves_toward_prior_as_tau_grows": bool(np.all(np.diff(prior_angles) < 1e-9)),
    "posterior_moves_away_from_sample_as_tau_grows": bool(np.all(np.diff(sample_angles) > -1e-9)),
}
shrinkage_checks



## Smoothing A Spherical Response

Smoothing a directional response means the fitted values must stay on the sphere. A simple extrinsic smoother averages nearby vectors in `R^3` and then projects the result back to unit length. This is not the only approach to spherical splines, but it exposes the minimum geometric responsibility: the fitted curve should be a path of directions.

The interactive artifact shows noisy observations, the latent curve, and the smoothed curve. Rotate it and inspect the path. The smoothed curve follows the trend without cutting through the sphere as a displayed object. The final checks verify unit length and compare angular error before and after smoothing.


In [ ]:

t = np.linspace(0.0, 1.0, 80)
latent = normalize(np.column_stack([
    0.75 * np.cos(1.6 * np.pi * t),
    0.75 * np.sin(1.6 * np.pi * t),
    -0.45 + 0.9 * t,
]))
# Generate visibly noisy directions around each latent direction, then smooth in the scalar predictor.
observed = np.vstack([sample_vmf(mu, 18.0, 1)[0] for mu in latent])

bandwidth = 0.035
smoothed = []
for ti in t:
    weights = np.exp(-0.5 * ((t - ti) / bandwidth) ** 2)
    smoothed.append(normalize((weights[:, None] * observed).sum(axis=0)))
smoothed = np.array(smoothed)
raw_error = np.degrees(angular_distance(observed, latent)).mean()
smooth_error = np.degrees(angular_distance(smoothed, latent)).mean()

sphere_u = np.linspace(0, 2*np.pi, 64)
sphere_v = np.linspace(0, np.pi, 32)
U, V = np.meshgrid(sphere_u, sphere_v)
SX = np.cos(U)*np.sin(V); SY = np.sin(U)*np.sin(V); SZ = np.cos(V)

curve_fig = go.Figure()
curve_fig.add_trace(go.Surface(x=SX, y=SY, z=SZ, opacity=0.16, showscale=False, colorscale=[[0, "#d9e3ea"], [1, "#d9e3ea"]], name="unit sphere"))
curve_fig.add_trace(go.Scatter3d(x=observed[:,0], y=observed[:,1], z=observed[:,2], mode="markers", marker=dict(size=4, color="#B84A62"), name="noisy directions"))
curve_fig.add_trace(go.Scatter3d(x=latent[:,0], y=latent[:,1], z=latent[:,2], mode="lines", line=dict(width=5, color="#2A9D8F"), name="latent curve"))
curve_fig.add_trace(go.Scatter3d(x=smoothed[:,0], y=smoothed[:,1], z=smoothed[:,2], mode="lines+markers", marker=dict(size=2, color="#F4A261"), line=dict(width=6, color="#F4A261"), name="smoothed curve"))
curve_fig.update_layout(
    title="Chapter 12 spherical smoothing lab: extrinsic local averaging with unit projection",
    scene=dict(xaxis_title="x", yaxis_title="y", zaxis_title="z", aspectmode="data"),
    margin=dict(l=0, r=0, t=50, b=0),
    height=640,
)
smoothing_path = save_plotly_html(curve_fig, TOPIC, "smoothing", "spherical-response-smoothing-lab.html", include_plotlyjs=True)
display_artifact(smoothing_path, width="100%", height=660)

smoothing_checks = {
    "observed_unit_norm_max_error": float(np.max(np.abs(np.linalg.norm(observed, axis=1) - 1.0))),
    "smoothed_unit_norm_max_error": float(np.max(np.abs(np.linalg.norm(smoothed, axis=1) - 1.0))),
    "smoothing_reduces_mean_angular_error": bool(smooth_error < raw_error),
    "raw_mean_error_degrees": float(raw_error),
    "smooth_mean_error_degrees": float(smooth_error),
}
smoothing_checks



## Modern-Method Checklist

Every method in this chapter has a Euclidean analogue, but the directional version needs its own checks.

- For **outliers**, measure angular influence on the resultant or fitted direction. Unit length prevents radial outliers, but not antipodal or off-cluster leverage.
- For **robust estimation**, state the angular loss or weighting rule. A robust estimate that is not a unit direction is not a directional estimate yet.
- For **bootstrap methods**, resample observations on the original sample space and summarize uncertainty as angles, caps, cones, or axes.
- For **density estimation**, integrate over the sphere or circle. A heatmap is only a density estimate if its mass is normalized on the curved space.
- For **Bayesian procedures**, interpret prior strength geometrically: prior direction plus pseudo-resultant is often the right first picture.
- For **smoothing**, keep fitted values on the manifold and report angular error or geodesic residuals.

The saved JSON file records these checks in the same spirit as the appendix table replacements: the methodology is modern, but the audit trail should be plain.


In [ ]:

numeric_checks = {
    **outlier_checks,
    **bootstrap_checks,
    **density_checks,
    **shrinkage_checks,
    **smoothing_checks,
    "source_span": source_span,
    "all_boolean_checks_pass": bool(
        all(value for value in outlier_checks.values() if isinstance(value, bool))
        and all(value for value in bootstrap_checks.values() if isinstance(value, bool))
        and all(value for value in density_checks.values() if isinstance(value, bool))
        and all(value for value in shrinkage_checks.values() if isinstance(value, bool))
        and all(value for value in smoothing_checks.values() if isinstance(value, bool))
    ),
}
assert numeric_checks["all_boolean_checks_pass"], numeric_checks
assert numeric_checks["density_integral_abs_error"] < 0.04
assert numeric_checks["smoothed_unit_norm_max_error"] < 1e-12
checks_path = save_json(numeric_checks, TOPIC, "checks", "modern-methodology-invariants.json")
display_artifact(checks_path)
numeric_checks



## Takeaways

Modern directional methodology is powerful because it keeps the geometry visible while relaxing old closed-form assumptions.

Outlier analysis asks which observations change the resultant or fitted direction, not which observations are far from the origin. Robust estimators encode an angular influence rule. Bootstrap confidence regions should be cones, caps, axes, or tangent regions that return to the sample space. Kernel density estimates must be normalized over surface area. Bayesian shrinkage is easiest to inspect as vector-resultant geometry before it becomes a posterior formula. Smoothing a spherical response should produce a curve on the sphere, with angular residuals used to evaluate the fit.

The chapter therefore sits at a useful boundary. It borrows resampling, robust statistics, density estimation, Bayesian ideas, and smoothing from broader statistics, but it keeps asking the same directional question: what transformation or constraint must the answer respect?


In [ ]:

final_sanity = {
    "source_span": source_span,
    "artifacts": assert_artifacts(
        [outlier_path, bootstrap_path, density_path, shrinkage_path, smoothing_path, checks_path],
        min_bytes=100,
    ),
    "core_checks": {
        "unit_input_data": outlier_checks["clean_unit_norm_max_error"] < 1e-12,
        "robust_direction_moves_less": bootstrap_checks["robust_moves_less_than_ordinary"],
        "bootstrap_cone_ordered": bootstrap_checks["cone_quantiles_ordered"],
        "density_integrates_near_one": density_checks["density_integral_abs_error"] < 0.04,
        "bayesian_shrinkage_moves_toward_prior": shrinkage_checks["posterior_moves_toward_prior_as_tau_grows"],
        "smoothing_stays_on_sphere": smoothing_checks["smoothed_unit_norm_max_error"] < 1e-12,
        "smoothing_reduces_error": smoothing_checks["smoothing_reduces_mean_angular_error"],
    },
    "pdf_used_for": "source orientation only; no copied prose, tables, screenshots, page crops, or figures",
    "standalone_contract": "modern methods are taught through angular influence, bootstrap cones, normalized density, shrinkage vectors, smoothing paths, and invariant checks",
}
assert all(final_sanity["core_checks"].values()), final_sanity
final_path = save_json(final_sanity, TOPIC, "checks", "final-sanity.json")
assert final_path.exists() and final_path.stat().st_size > 20
final_sanity
